# Notebook 7 — Clinical Report Generator & Final Analysis
**RSNA Intracranial Hemorrhage Detection**

This notebook ties together all pipeline components to generate structured
screening reports for individual CT brain images, and provides the final
analysis sections required for a complete project deliverable.

### Report schema (fixed fields, locked phrases)
Each report contains:
- **Screening outcome** — fixed vocabulary, no diagnostic claim
- **Calibrated confidence** — numeric probability + triage band
- **Recommended action** — one of three fixed phrases
- **Grad-CAM overlay** — visual evidence with caveats
- **Disclaimer** — regulatory/safety language

### Final analysis sections
- **Patient-level aggregation** — aggregate slice predictions to study/patient level
- **Training stability summary** — convergence evidence across final epochs
- **Capacity ceiling reflection** — architectural limits and future directions
- **Deployment decision** — final model selection and configuration

### Required inputs
- NB02 output: `manifest.csv` + `cache/` NPY arrays
- NB03 output: `best_model.pth`, `checkpoint.pth`, `training_metrics.csv`
- NB06 output: `calibration_params.json`

> The report generator is deterministic — same image always produces the same report.

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────
import os, json, base64, datetime, uuid, glob as _glob_mod
from io import BytesIO
from pathlib import Path
import numpy as np
import pandas as pd
import cv2                                  # still needed for cv2.resize in overlay + cv2.imwrite
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as T
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from sklearn.metrics import (
    roc_auc_score, roc_curve, confusion_matrix,
    precision_recall_curve, brier_score_loss
)
from IPython.display import display, HTML

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# ── Input paths ──────────────────────────────────────────────────────────
CACHE_INPUT_DIR   = '/kaggle/input/notebooks/harshitghosh/nb02eda'
NPY_CACHE_DIR     = f'{CACHE_INPUT_DIR}/cache'
MANIFEST_PATH     = f'{CACHE_INPUT_DIR}/manifest.csv'
MODEL_PATH        = '/kaggle/input/notebooks/harshitghosh/03nbeda/best_model.pth'
CHECKPOINT_PATH   = '/kaggle/input/notebooks/harshitghosh/03nbeda/checkpoint.pth'
METRICS_LOG_PATH  = '/kaggle/input/notebooks/harshitghosh/03nbeda/training_metrics.csv'
CALIB_PARAMS_PATH = '/kaggle/input/notebooks/harshitghosh/06-calibration/calibration_params.json'

ARCH        = 'efficientnet_b0'
IMG_SIZE    = 256
SEED        = 42
N_REPORTS   = 10      # number of sample reports to generate and display

REPORTS_DIR = Path('/kaggle/working/reports')
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# ─── Load normalization stats ────────────────────────────────────────────
_norm_path = os.path.join(CACHE_INPUT_DIR, 'normalization_stats.json')
if os.path.exists(_norm_path):
    with open(_norm_path) as f:
        _norm = json.load(f)
    MEAN = _norm['mean']
    STD  = _norm['std']
    print(f'Dataset normalization: mean={MEAN}, std={STD}')
else:
    MEAN = [0.485, 0.456, 0.406]
    STD  = [0.229, 0.224, 0.225]
    print(f'Using ImageNet defaults: mean={MEAN}, std={STD}')

print(f'Device: {DEVICE}')

In [ ]:
# ── 1. Load calibration parameters ───────────────────────────────────────
with open(CALIB_PARAMS_PATH) as f:
    calib = json.load(f)

TEMPERATURE     = calib['temperature']
# Use calibrated threshold (re-optimised after temp scaling) if available,
# otherwise fall back to the raw Youden threshold
BASE_THRESHOLD  = calib.get('calibrated_threshold', calib['base_threshold'])
HIGH_THRESHOLD  = calib['high_threshold']
LOW_THRESHOLD   = calib['low_threshold']
CALIB_METHOD    = calib.get('method', 'temperature')

raw_ece = calib.get('raw_ece', None)
cal_ece = calib.get('cal_ece', None)
hc_ece_raw = calib.get('hc_ece_raw', None)
hc_ece_cal = calib.get('hc_ece_cal', None)

print('Calibration parameters:')
print(json.dumps(calib, indent=2))
print(f'\nDeployment calibrator : {CALIB_METHOD}')
print(f'Decision threshold    : {BASE_THRESHOLD:.4f} (calibrated)')

# ── Calibration honest assessment ────────────────────────────────────────
print(f'\n{"─" * 55}')
print('CALIBRATION EFFECT SUMMARY')
print(f'{"─" * 55}')
if raw_ece is not None and cal_ece is not None:
    ece_delta = cal_ece - raw_ece
    direction = 'WORSENED' if ece_delta > 0 else 'IMPROVED'
    print(f'  Global ECE     : {raw_ece:.5f} → {cal_ece:.5f} ({direction}, Δ={ece_delta:+.5f})')
if hc_ece_raw is not None and hc_ece_cal is not None:
    hc_delta = hc_ece_cal - hc_ece_raw
    hc_dir = 'IMPROVED' if hc_delta < 0 else 'WORSENED'
    print(f'  HC-band ECE    : {hc_ece_raw:.5f} → {hc_ece_cal:.5f} ({hc_dir}, Δ={hc_delta:+.5f})')
raw_brier = calib.get('raw_brier', None)
cal_brier = calib.get('cal_brier', None)
if raw_brier is not None and cal_brier is not None:
    brier_delta = cal_brier - raw_brier
    brier_dir = 'IMPROVED' if brier_delta < 0 else 'WORSENED'
    print(f'  Brier score    : {raw_brier:.5f} → {cal_brier:.5f} ({brier_dir}, Δ={brier_delta:+.5f})')
print(f'\n  Assessment:')
print(f'    Temperature scaling (T={TEMPERATURE:.4f}) WORSENED global ECE slightly')
print(f'    (+0.026) but IMPROVED high-confidence band ECE by {abs(hc_ece_cal - hc_ece_raw):.3f}')
print(f'    and Brier score by {abs(cal_brier - raw_brier):.4f}.')
print(f'    For triage deployment, high-confidence calibration is the critical')
print(f'    metric — patients routed to URGENT review must have reliable')
print(f'    probability estimates. Global ECE regression is acceptable.')
print(f'{"─" * 55}')

In [ ]:
# ── 2. Load model ─────────────────────────────────────────────────────────
def build_model(arch):
    if arch == 'efficientnet_b0':
        m = models.efficientnet_b0(weights=None)
        m.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(m.classifier[1].in_features, 1))
    elif arch == 'resnet50':
        m = models.resnet50(weights=None)
        m.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(m.fc.in_features, 1))
    return m


model = build_model(ARCH)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model = model.to(DEVICE)
print('Model loaded.')

In [ ]:
# ── 3. Grad-CAM ───────────────────────────────────────────────────────────
class GradCAM:
    def __init__(self, model, arch):
        self.model = model
        self.activations = self.gradients = None
        target = model.features[-1] if arch == 'efficientnet_b0' else model.layer4[-1]
        self._fh = target.register_forward_hook(lambda m, i, o: setattr(self, 'activations', o.detach()))
        self._bh = target.register_full_backward_hook(lambda m, gi, go: setattr(self, 'gradients', go[0].detach()))

    def remove(self):
        self._fh.remove(); self._bh.remove()

    def generate(self, img_tensor):
        self.model.zero_grad()
        t = img_tensor.clone().requires_grad_(True)
        self.model(t).squeeze().backward()
        w   = self.gradients.squeeze().mean(dim=(1, 2), keepdim=True)
        cam = torch.relu((w * self.activations.squeeze()).sum(dim=0)).cpu().numpy()
        if cam.max() > 0:
            cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam


grad_cam = GradCAM(model, ARCH)
val_transform = T.Compose([T.ToPILImage(), T.ToTensor(), T.Normalize(mean=MEAN, std=STD)])


def load_npy(image_id):
    """Load a cached NPY array and return uint8 RGB for display / overlay."""
    path = os.path.join(NPY_CACHE_DIR, f'{image_id}.npy')
    try:
        return np.load(path)              # uint8  [0, 255]
    except Exception:
        return np.zeros((IMG_SIZE, IMG_SIZE, 3), np.uint8)


def get_tensor(image_id):
    return val_transform(load_npy(image_id)).unsqueeze(0).to(DEVICE)


def make_overlay(orig_rgb, cam, alpha=0.45):
    cam_r = cv2.resize(cam, (orig_rgb.shape[1], orig_rgb.shape[0]))
    heat  = (cm.jet(cam_r)[:, :, :3] * 255).astype(np.uint8)
    return (alpha * heat + (1 - alpha) * orig_rgb).astype(np.uint8)


print('Grad-CAM & helpers ready.')

In [ ]:
# ── 4. Inference + calibration pipeline ──────────────────────────────────
def infer_single(image_id: str) -> dict:
    """
    Run forward pass, apply temperature calibration, return prediction dict.
    """
    model.train()   # train mode so Grad-CAM gradients flow
    t = get_tensor(image_id)
    with torch.no_grad():
        logit = model(t).squeeze().cpu().item()

    raw_prob  = float(torch.sigmoid(torch.tensor(logit)).item())
    cal_prob  = float(torch.sigmoid(torch.tensor(logit / TEMPERATURE)).item())

    model.train()
    cam = grad_cam.generate(t)
    model.eval()

    return {'logit': logit, 'raw_prob': raw_prob, 'cal_prob': cal_prob, 'cam': cam}


print('Inference pipeline defined.')

In [ ]:
# ── 5. Report schema (fixed vocabulary) ──────────────────────────────────
"""
FIXED VOCABULARY — do not change these strings.
The report module is intentionally restricted to prevent diagnostic claims.
"""

# Outcome phrases (based on threshold)
OUTCOME_POSITIVE = 'Hemorrhage indicator detected'
OUTCOME_NEGATIVE = 'No hemorrhage indicator detected'

# Band labels
BAND_LABELS = {
    'HIGH'  : 'High confidence',
    'MEDIUM': 'Moderate confidence',
    'LOW'   : 'Low confidence',
}

# Triage action phrases (one per band × outcome combination)
TRIAGE_ACTIONS = {
    ('POSITIVE', 'HIGH')  : 'Urgent radiologist review recommended',
    ('POSITIVE', 'MEDIUM'): 'Prioritised radiologist review recommended',
    ('POSITIVE', 'LOW')   : 'Radiologist review recommended',
    ('NEGATIVE', 'HIGH')  : 'Standard workflow — no immediate escalation indicated',
    ('NEGATIVE', 'MEDIUM'): 'Standard workflow — manual review if clinically indicated',
    ('NEGATIVE', 'LOW')   : 'Standard workflow',
}

DISCLAIMER = (
    'This report is produced by an AI-assisted screening tool and does NOT constitute a '
    'medical diagnosis. All screening findings must be reviewed and confirmed by a '
    'qualified, licensed medical professional before any clinical decision is made. '
    'The system is intended solely as a decision-support aid in a screening workflow '
    'and is not cleared for standalone diagnostic use.'
)


def build_report(image_id: str, inference: dict,
                 true_label: int = None) -> dict:
    """
    Build a structured screening report from inference results.
    Enforces fixed vocabulary — never outputs free-form clinical text.
    """
    cal_prob = inference['cal_prob']

    # Band assignment
    if cal_prob >= HIGH_THRESHOLD:
        band = 'HIGH'
    elif cal_prob >= LOW_THRESHOLD:
        band = 'MEDIUM'
    else:
        band = 'LOW'

    # Outcome: based on base_threshold (Youden-optimal from training)
    is_positive = cal_prob >= BASE_THRESHOLD
    outcome_str = OUTCOME_POSITIVE if is_positive else OUTCOME_NEGATIVE
    outcome_key = 'POSITIVE' if is_positive else 'NEGATIVE'

    triage_action = TRIAGE_ACTIONS[(outcome_key, band)]

    # Save Grad-CAM overlay
    orig_rgb = load_npy(image_id)
    overlay  = make_overlay(orig_rgb, inference['cam'])
    cam_save_path = str(REPORTS_DIR / f'{image_id}_gradcam.png')
    cv2.imwrite(cam_save_path, cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))

    report = {
        'report_id'        : f'RPT_{datetime.datetime.now().strftime("%Y%m%d_%H%M%S")}_{image_id[-8:]}',
        'generated_at'     : datetime.datetime.utcnow().isoformat() + 'Z',
        'image_id'         : image_id,
        'ground_truth_label': int(true_label) if true_label is not None else 'N/A',
        'screening_module' : {
            'version'             : '1.0',
            'architecture'        : ARCH,
            'calibration_method'  : CALIB_METHOD,
        },
        'prediction': {
            'screening_outcome'      : outcome_str,
            'raw_probability'        : round(inference['raw_prob'], 4),
            'calibrated_probability' : round(cal_prob, 4),
            'confidence_band'        : band,
            'confidence_band_label'  : BAND_LABELS[band],
            'decision_threshold'     : round(BASE_THRESHOLD, 4),
        },
        'triage': {
            'action'  : triage_action,
            'urgency' : 'URGENT' if (is_positive and band == 'HIGH') else 'STANDARD',
        },
        'explainability': {
            'method'     : 'Gradient-weighted Class Activation Mapping (Grad-CAM)',
            'heatmap_path': cam_save_path,
            'note'       : (
                'Highlighted regions indicate areas with greatest influence on the '
                'screening decision. These are not confirmed anatomical findings.'
            ),
        },
        'disclaimer': DISCLAIMER,
    }
    return report


print('Report schema defined.')

In [ ]:
# ── 6. Generate reports for sample images ────────────────────────────────
import random
random.seed(SEED)

manifest = pd.read_csv(MANIFEST_PATH)
val_df   = manifest[manifest['split'] == 'val'].reset_index(drop=True)

# Sample a mix: 5 positive + 5 negative
pos_samples = val_df[val_df['any'] == 1].sample(N_REPORTS // 2, random_state=SEED)
neg_samples = val_df[val_df['any'] == 0].sample(N_REPORTS // 2, random_state=SEED)
sample_df   = pd.concat([pos_samples, neg_samples]).sample(frac=1, random_state=SEED)

all_reports = []
for _, row in sample_df.iterrows():
    inf = infer_single(row['image_id'])
    rep = build_report(row['image_id'], inf, true_label=row['any'])
    all_reports.append(rep)

    # Save individual JSON report
    report_path = REPORTS_DIR / f'{row["image_id"]}_report.json'
    with open(report_path, 'w') as f:
        json.dump({k: v for k, v in rep.items() if k != 'cam'}, f, indent=2)

print(f'{len(all_reports)} reports generated and saved to {REPORTS_DIR}')

In [ ]:
# ── 7. HTML report display ────────────────────────────────────────────────
def img_to_base64(path: str) -> str:
    with open(path, 'rb') as f:
        return base64.b64encode(f.read()).decode('utf-8')


BAND_COLORS = {'HIGH': '#e74c3c', 'MEDIUM': '#f39c12', 'LOW': '#27ae60'}
URGENCY_COLORS = {'URGENT': '#e74c3c', 'STANDARD': '#2980b9'}


def render_report_html(report: dict) -> str:
    pred   = report['prediction']
    triage = report['triage']
    expl   = report['explainability']
    band   = pred['confidence_band']
    band_color = BAND_COLORS.get(band, '#555')
    urg_color  = URGENCY_COLORS.get(triage['urgency'], '#555')

    # Embed Grad-CAM image as base64
    cam_data = img_to_base64(expl['heatmap_path']) if os.path.exists(expl['heatmap_path']) else ''
    cam_img_tag = f'<img src="data:image/png;base64,{cam_data}" style="height:200px;"/>' if cam_data else '<i>Image not available</i>'

    outcome_color = '#e74c3c' if 'detected' in pred['screening_outcome'].lower() and 'No' not in pred['screening_outcome'] else '#27ae60'

    html = f"""
<div style="border:1px solid #ccc; border-radius:8px; padding:16px; margin:16px 0;
             font-family:Arial,sans-serif; max-width:800px; background:#fafafa;">
  <h3 style="margin:0 0 8px;">Screening Report
    <span style="font-size:0.7em; color:#888; float:right;">{report['report_id']}</span>
  </h3>
  <p style="margin:2px 0; color:#555; font-size:0.85em;">
    Image ID: <code>{report['image_id']}</code> &nbsp;|
    Generated: {report['generated_at']} &nbsp;|
    Ground truth: <b>{'Positive' if report['ground_truth_label'] == 1 else 'Negative' if report['ground_truth_label'] == 0 else 'Unknown'}</b>
  </p>
  <hr style="margin:10px 0;"/>

  <table style="width:100%; border-collapse:collapse;">
    <tr>
      <td style="width:50%; vertical-align:top; padding-right:16px;">
        <h4 style="margin:0 0 8px;">Screening Outcome</h4>
        <div style="background:{outcome_color}; color:white; padding:8px 12px;
                    border-radius:4px; font-size:1.05em; font-weight:bold;">
          {pred['screening_outcome']}
        </div>

        <h4 style="margin:12px 0 6px;">Confidence</h4>
        <table style="font-size:0.9em; width:100%;">
          <tr><td>Raw probability:</td><td><b>{pred['raw_probability']:.4f}</b></td></tr>
          <tr><td>Calibrated probability:</td><td><b>{pred['calibrated_probability']:.4f}</b></td></tr>
          <tr><td>Decision threshold:</td><td><b>{pred['decision_threshold']:.4f}</b></td></tr>
          <tr><td>Confidence band:</td>
              <td><span style="background:{band_color}; color:white; padding:2px 8px;
                               border-radius:3px; font-weight:bold;">
                {pred['confidence_band_label']} ({band})
              </span></td></tr>
        </table>

        <h4 style="margin:12px 0 6px;">Triage Recommendation</h4>
        <div style="border-left:4px solid {urg_color}; padding-left:10px;">
          <span style="color:{urg_color}; font-weight:bold;">{triage['urgency']}</span><br/>
          {triage['action']}
        </div>
      </td>

      <td style="width:50%; vertical-align:top;">
        <h4 style="margin:0 0 8px;">Visual Evidence (Grad-CAM)</h4>
        {cam_img_tag}
        <p style="font-size:0.8em; color:#888; margin:4px 0;">{expl['note']}</p>
      </td>
    </tr>
  </table>

  <hr style="margin:12px 0;"/>
  <p style="font-size:0.75em; color:#888; margin:0; border-left:3px solid #e74c3c; padding-left:8px;">
    <b>DISCLAIMER:</b> {report['disclaimer']}
  </p>
</div>
"""
    return html


print('HTML renderer defined. Displaying reports...')

In [ ]:
# ── 8. Render all sample reports ──────────────────────────────────────────
for report in all_reports:
    display(HTML(render_report_html(report)))

In [ ]:
# ── 9. Batch summary table + Triage Risk Analysis ─────────────────────────
summary_rows = []
for rep in all_reports:
    pred = rep['prediction']
    summary_rows.append({
        'image_id'           : rep['image_id'][-8:],
        'true_label'         : rep['ground_truth_label'],
        'screening_outcome'  : pred['screening_outcome'],
        'cal_prob'           : pred['calibrated_probability'],
        'confidence_band'    : pred['confidence_band'],
        'triage_action'      : rep['triage']['action'],
        'urgency'            : rep['triage']['urgency'],
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv('/kaggle/working/report_summary.csv', index=False)

print('Batch summary:')
display(summary_df)

# ── Triage Risk Analysis ─────────────────────────────────────────────────
# Go beyond "did we escalate positives?" — also analyze false escalations.
is_positive_pred = summary_df['screening_outcome'].str.contains('detected') & \
                   ~summary_df['screening_outcome'].str.contains('No ')
is_positive_true = summary_df['true_label'] == 1
is_urgent = summary_df['urgency'] == 'URGENT'

tp = (is_positive_pred & is_positive_true).sum()
fp = (is_positive_pred & ~is_positive_true).sum()
fn = (~is_positive_pred & is_positive_true).sum()
tn = (~is_positive_pred & ~is_positive_true).sum()
n_pos = is_positive_true.sum()
n_neg = (~is_positive_true).sum()

# Urgent escalation analysis
urgent_tp = (is_urgent & is_positive_true).sum()
urgent_fp = (is_urgent & ~is_positive_true).sum()
urgent_fn = (~is_urgent & is_positive_true).sum()

print(f'\n{"═" * 55}')
print(f'TRIAGE RISK ANALYSIS (n={len(summary_df)} sample reports)')
print(f'{"═" * 55}')
print(f'\n  Screening Decision:')
print(f'    True Positives  : {tp}/{n_pos} hemorrhage cases detected')
print(f'    False Positives : {fp}/{n_neg} normal cases falsely flagged')
print(f'    False Negatives : {fn}/{n_pos} hemorrhage cases missed')
print(f'    True Negatives  : {tn}/{n_neg} normal cases correctly cleared')
print(f'    Precision (PPV) : {tp / max(tp + fp, 1):.2%}')
print(f'    Recall (Sens)   : {tp / max(tp + fn, 1):.2%}')

print(f'\n  Urgent Escalation:')
print(f'    True urgent     : {urgent_tp} (hemorrhage → URGENT)')
print(f'    False urgent    : {urgent_fp} (normal → URGENT) ← unnecessary escalation')
print(f'    Missed urgent   : {urgent_fn} (hemorrhage → not URGENT)')
print(f'    Escalation PPV  : {urgent_tp / max(urgent_tp + urgent_fp, 1):.2%}')

if fp > 0:
    print(f'\n  ⚠ FALSE POSITIVE DETAIL:')
    fp_cases = summary_df[is_positive_pred & ~is_positive_true]
    for _, row in fp_cases.iterrows():
        print(f'    {row["image_id"]}  cal_prob={row["cal_prob"]:.4f}  '
              f'band={row["confidence_band"]}  action="{row["triage_action"]}"')
    print(f'\n    These represent unnecessary radiologist escalations.')
    print(f'    In deployment, {fp} of {fp + tp} flagged cases are false alarms.')

if fn > 0:
    print(f'\n  ⚠ FALSE NEGATIVE DETAIL:')
    fn_cases = summary_df[~is_positive_pred & is_positive_true]
    for _, row in fn_cases.iterrows():
        print(f'    {row["image_id"]}  cal_prob={row["cal_prob"]:.4f}  '
              f'band={row["confidence_band"]}  → MISSED')
    print(f'    These hemorrhage cases would not be escalated.')

print(f'\n  Note: This analysis is on {len(summary_df)} sampled images.')
print(f'  Full-set triage metrics are computed in the patient-level section below.')
print(f'{"═" * 55}')

---
# Part II — Final Analysis

The sections below address key structural requirements for a complete project deliverable:
patient-level aggregation, training stability, capacity analysis, and deployment configuration.

In [ ]:
# ── 10. Patient-Level Aggregation ──────────────────────────────────────────
# All training, calibration, and evaluation so far has been at SLICE level.
# Clinically and competition-wise, the PATIENT (study) is the unit.
# We aggregate slice probabilities to patient level using multiple strategies.

from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

# ── Load full validation set and run inference ───────────────────────────
full_manifest = pd.read_csv(MANIFEST_PATH)
val_full = full_manifest[full_manifest['split'] == 'val'].reset_index(drop=True)

print(f'Validation set: {len(val_full):,} slices, '
      f'{val_full["patient_id"].nunique():,} patients')


class SimpleDataset(Dataset):
    def __init__(self, df, npy_root, transform):
        self.df = df.reset_index(drop=True)
        self.npy_root = npy_root
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = os.path.join(self.npy_root, f'{row["image_id"]}.npy')
        try:
            img = np.load(path)
        except Exception:
            img = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        return self.transform(img), idx


val_dataset = SimpleDataset(val_full, NPY_CACHE_DIR, val_transform)
val_loader  = DataLoader(val_dataset, batch_size=64, shuffle=False,
                         num_workers=4, pin_memory=True)

# ── Collect raw logits for all val slices ─────────────────────────────────
model.eval()
all_logits = np.zeros(len(val_full))

with torch.no_grad():
    for batch_imgs, batch_idxs in tqdm(val_loader, desc='Val inference'):
        logits = model(batch_imgs.to(DEVICE)).squeeze(-1).cpu().numpy()
        for i, idx in enumerate(batch_idxs.numpy()):
            all_logits[idx] = logits[i]

raw_probs = 1 / (1 + np.exp(-all_logits))                     # sigmoid
cal_probs = 1 / (1 + np.exp(-all_logits / TEMPERATURE))       # temp-scaled

val_full['raw_prob'] = raw_probs
val_full['cal_prob'] = cal_probs
val_full['true_label'] = val_full['any'].astype(int)

# ── Slice-level AUC (sanity check) ──────────────────────────────────────
slice_auc = roc_auc_score(val_full['true_label'], val_full['cal_prob'])
print(f'\nSlice-level AUC (calibrated): {slice_auc:.5f}')

# ── Patient-level aggregation: max, mean, noisy-or ──────────────────────
def noisy_or(probs):
    """Noisy-OR: P(any) = 1 - prod(1 - p_i)"""
    return 1.0 - np.prod(1.0 - probs)


patient_agg = val_full.groupby('patient_id').agg(
    true_label = ('true_label', 'max'),   # patient positive if ANY slice is positive
    n_slices   = ('true_label', 'count'),
    prob_max   = ('cal_prob', 'max'),
    prob_mean  = ('cal_prob', 'mean'),
    prob_noisy_or = ('cal_prob', noisy_or),
).reset_index()

print(f'\nPatient-level summary: {len(patient_agg):,} patients')
print(f'  Positive patients: {patient_agg["true_label"].sum():,} '
      f'({patient_agg["true_label"].mean()*100:.1f}%)')
print(f'  Median slices/patient: {patient_agg["n_slices"].median():.0f}')

# ── Patient-level AUC for each aggregation strategy ─────────────────────
strategies = {
    'Max (slice)'   : 'prob_max',
    'Mean (slice)'  : 'prob_mean',
    'Noisy-OR'      : 'prob_noisy_or',
}

print(f'\nPatient-Level AUC by Aggregation Strategy:')
print(f'  {"Strategy":<16s}  {"AUC":>8s}  {"Sens":>6s}  {"Spec":>6s}')
print(f'  {"─"*16}  {"─"*8}  {"─"*6}  {"─"*6}')

best_patient_strategy = None
best_patient_auc = 0

for name, col in strategies.items():
    auc = roc_auc_score(patient_agg['true_label'], patient_agg[col])
    fpr, tpr, thresholds = roc_curve(patient_agg['true_label'], patient_agg[col])
    j_idx = np.argmax(tpr - fpr)
    thr = thresholds[j_idx]
    preds = (patient_agg[col] >= thr).astype(int)
    tp = ((preds == 1) & (patient_agg['true_label'] == 1)).sum()
    fn = ((preds == 0) & (patient_agg['true_label'] == 1)).sum()
    tn = ((preds == 0) & (patient_agg['true_label'] == 0)).sum()
    fp = ((preds == 1) & (patient_agg['true_label'] == 0)).sum()
    sens = tp / max(tp + fn, 1)
    spec = tn / max(tn + fp, 1)
    print(f'  {name:<16s}  {auc:>8.5f}  {sens:>6.4f}  {spec:>6.4f}')

    if auc > best_patient_auc:
        best_patient_auc = auc
        best_patient_strategy = name
        best_patient_thr = thr
        best_patient_col = col

print(f'\n★ Best patient-level strategy: {best_patient_strategy} '
      f'(AUC={best_patient_auc:.5f})')
print(f'  vs. slice-level AUC: {slice_auc:.5f} '
      f'(Δ = {best_patient_auc - slice_auc:+.5f})')

In [ ]:
# ── 11. Patient-Level ROC Curve & Confusion Matrix ────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── ROC curves: slice vs patient (all strategies) ────────────────────────
# Slice-level ROC
s_fpr, s_tpr, _ = roc_curve(val_full['true_label'], val_full['cal_prob'])
axes[0].plot(s_fpr, s_tpr, 'k--', linewidth=1.5, alpha=0.5,
             label=f'Slice-level (AUC={slice_auc:.4f})')

colors = {'Max (slice)': 'tab:blue', 'Mean (slice)': 'tab:orange', 'Noisy-OR': 'tab:green'}
for name, col in strategies.items():
    p_fpr, p_tpr, _ = roc_curve(patient_agg['true_label'], patient_agg[col])
    p_auc = roc_auc_score(patient_agg['true_label'], patient_agg[col])
    lw = 2.5 if name == best_patient_strategy else 1.5
    axes[0].plot(p_fpr, p_tpr, color=colors[name], linewidth=lw,
                 label=f'{name} (AUC={p_auc:.4f})')

axes[0].plot([0, 1], [0, 1], 'k:', linewidth=0.5)
axes[0].set(title='ROC: Slice-Level vs Patient-Level',
            xlabel='False Positive Rate', ylabel='True Positive Rate')
axes[0].legend(fontsize=8)
axes[0].set_xlim(0, 1); axes[0].set_ylim(0, 1)

# ── Confusion matrix (best patient-level strategy) ───────────────────────
best_preds = (patient_agg[best_patient_col] >= best_patient_thr).astype(int)
cm = confusion_matrix(patient_agg['true_label'], best_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=['Pred Neg', 'Pred Pos'],
            yticklabels=['True Neg', 'True Pos'])
axes[1].set_title(f'Patient-Level Confusion Matrix\n({best_patient_strategy}, '
                  f'thr={best_patient_thr:.3f})')

# ── Distribution of max slice probability per patient ────────────────────
pos_patients = patient_agg[patient_agg['true_label'] == 1]['prob_max']
neg_patients = patient_agg[patient_agg['true_label'] == 0]['prob_max']
axes[2].hist(neg_patients, bins=40, alpha=0.6, color='tab:blue',
             label=f'Negative (n={len(neg_patients)})', density=True)
axes[2].hist(pos_patients, bins=40, alpha=0.6, color='tab:red',
             label=f'Positive (n={len(pos_patients)})', density=True)
axes[2].axvline(best_patient_thr, color='black', linestyle='--',
                label=f'Threshold={best_patient_thr:.3f}')
axes[2].set(title='Patient Max-Probability Distribution',
            xlabel='Max calibrated probability', ylabel='Density')
axes[2].legend(fontsize=8)

plt.suptitle('Patient-Level Analysis', fontsize=13)
plt.tight_layout()
plt.savefig('/kaggle/working/patient_level_analysis.png', bbox_inches='tight')
plt.show()

# ── Patient-level performance summary ────────────────────────────────────
tp = cm[1, 1]; fn = cm[1, 0]; tn = cm[0, 0]; fp = cm[0, 1]
sens = tp / max(tp + fn, 1)
spec = tn / max(tn + fp, 1)
ppv  = tp / max(tp + fp, 1)
npv  = tn / max(tn + fn, 1)
f1   = 2 * tp / max(2 * tp + fp + fn, 1)

print(f'\n{"═" * 50}')
print(f'PATIENT-LEVEL PERFORMANCE ({best_patient_strategy})')
print(f'{"═" * 50}')
print(f'  AUC         : {best_patient_auc:.5f}')
print(f'  Sensitivity : {sens:.4f}')
print(f'  Specificity : {spec:.4f}')
print(f'  PPV         : {ppv:.4f}')
print(f'  NPV         : {npv:.4f}')
print(f'  F1          : {f1:.4f}')
print(f'  Threshold   : {best_patient_thr:.4f}')
print(f'{"═" * 50}')

# ── Patient-level triage risk (full validation set) ──────────────────────
print(f'\n{"═" * 50}')
print(f'PATIENT-LEVEL TRIAGE RISK (full val set)')
print(f'{"═" * 50}')
print(f'  Total patients       : {len(patient_agg):,}')
print(f'  Positive (hemorrhage): {int(patient_agg["true_label"].sum()):,}')
print(f'  Negative (normal)    : {int((patient_agg["true_label"] == 0).sum()):,}')
print(f'\n  At threshold {best_patient_thr:.4f}:')
print(f'    True Positives     : {tp:,}')
print(f'    False Positives    : {fp:,} ← unnecessary escalations')
print(f'    False Negatives    : {fn:,} ← missed hemorrhages')
print(f'    True Negatives     : {tn:,}')
print(f'    Escalation PPV     : {ppv:.4f} ({ppv*100:.1f}% of flagged patients have hemorrhage)')
print(f'    Miss rate (FNR)    : {fn / max(fn + tp, 1):.4f} ({fn / max(fn + tp, 1)*100:.1f}% of hemorrhage patients missed)')
print(f'{"═" * 50}')

# Save patient-level results
patient_results = {
    'aggregation_strategy': best_patient_strategy,
    'patient_auc':    round(best_patient_auc, 5),
    'slice_auc':      round(slice_auc, 5),
    'sensitivity':    round(sens, 4),
    'specificity':    round(spec, 4),
    'ppv':            round(ppv, 4),
    'npv':            round(npv, 4),
    'f1':             round(f1, 4),
    'threshold':      round(best_patient_thr, 4),
    'n_patients':     len(patient_agg),
    'n_positive':     int(patient_agg['true_label'].sum()),
    'false_positives': int(fp),
    'false_negatives': int(fn),
    'escalation_ppv':  round(ppv, 4),
    'miss_rate_fnr':   round(fn / max(fn + tp, 1), 4),
}
with open('/kaggle/working/patient_level_results.json', 'w') as f:
    json.dump(patient_results, f, indent=2)

print('\nSaved: patient_level_results.json, patient_level_analysis.png')

In [ ]:
# ── 11b. Patient-Level Recalibration ──────────────────────────────────────
# Slice-level calibration does NOT transfer after max-pooling aggregation.
# Max-pooling inflates probabilities → the calibration mapping is invalid.
# We apply isotonic regression with 5-fold CV at the patient level.

from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import KFold

patient_probs  = patient_agg[best_patient_col].values
patient_labels = patient_agg['true_label'].values

# ── ECE helper ───────────────────────────────────────────────────────────
def compute_ece(probs, labels, n_bins=15):
    """Expected Calibration Error with equal-width bins."""
    bin_edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        mask = (probs >= bin_edges[i]) & (probs < bin_edges[i + 1])
        if mask.sum() == 0:
            continue
        bin_acc  = labels[mask].mean()
        bin_conf = probs[mask].mean()
        ece += mask.sum() * abs(bin_acc - bin_conf)
    return ece / len(probs)

# ── Pre-recalibration ECE at patient level ───────────────────────────────
patient_ece_before = compute_ece(patient_probs, patient_labels)

# ── 5-fold CV isotonic regression at patient level ───────────────────────
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
recal_probs = np.zeros_like(patient_probs)

for fold_idx, (train_idx, val_idx) in enumerate(kf.split(patient_probs)):
    ir_fold = IsotonicRegression(out_of_bounds='clip')
    ir_fold.fit(patient_probs[train_idx], patient_labels[train_idx])
    recal_probs[val_idx] = ir_fold.predict(patient_probs[val_idx])

patient_ece_after = compute_ece(recal_probs, patient_labels)

# ── Full-data isotonic for deployment ────────────────────────────────────
ir_patient = IsotonicRegression(out_of_bounds='clip')
ir_patient.fit(patient_probs, patient_labels)

# ── Re-optimise threshold after patient-level recalibration ──────────────
fpr_r, tpr_r, thr_r = roc_curve(patient_labels, recal_probs)
j_idx_r = np.argmax(tpr_r - fpr_r)
recal_thr = float(thr_r[j_idx_r])
recal_auc = roc_auc_score(patient_labels, recal_probs)

recal_preds = (recal_probs >= recal_thr).astype(int)
cm_r = confusion_matrix(patient_labels, recal_preds)
tp_r = cm_r[1, 1]; fn_r = cm_r[1, 0]; tn_r = cm_r[0, 0]; fp_r = cm_r[0, 1]
sens_r = tp_r / max(tp_r + fn_r, 1)
spec_r = tn_r / max(tn_r + fp_r, 1)
ppv_r  = tp_r / max(tp_r + fp_r, 1)
npv_r  = tn_r / max(tn_r + fn_r, 1)
f1_r   = 2 * tp_r / max(2 * tp_r + fp_r + fn_r, 1)

# ── Reliability diagram: before vs after recalibration ───────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

def plot_rel_diagram(ax, probs, labels, title, ece_val, n_bins=15):
    bin_edges = np.linspace(0, 1, n_bins + 1)
    bin_confs, bin_accs, bin_sizes = [], [], []
    for i in range(n_bins):
        mask = (probs >= bin_edges[i]) & (probs < bin_edges[i + 1])
        if mask.sum() > 0:
            bin_confs.append(probs[mask].mean())
            bin_accs.append(labels[mask].mean())
            bin_sizes.append(mask.sum())
        else:
            bin_confs.append((bin_edges[i] + bin_edges[i+1]) / 2)
            bin_accs.append(0)
            bin_sizes.append(0)
    bc, ba, bs = np.array(bin_confs), np.array(bin_accs), np.array(bin_sizes)
    mask = bs > 0
    ax.bar(bc[mask], ba[mask], width=1.0/n_bins, alpha=0.6, color='tab:blue', align='center')
    ax.plot([0, 1], [0, 1], 'k--', linewidth=1)
    ax.set(title=f'{title}\nECE={ece_val:.4f}', xlabel='Mean predicted prob', ylabel='Fraction positive')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)

plot_rel_diagram(axes[0], patient_probs, patient_labels,
                 'Before Recalibration', patient_ece_before)
plot_rel_diagram(axes[1], recal_probs, patient_labels,
                 'After Recalibration (Isotonic 5-fold CV)', patient_ece_after)

# Confusion matrix after recalibration
sns.heatmap(cm_r, annot=True, fmt='d', cmap='Greens', ax=axes[2],
            xticklabels=['Pred Neg', 'Pred Pos'],
            yticklabels=['True Neg', 'True Pos'])
axes[2].set_title(f'Recalibrated Confusion Matrix\n(thr={recal_thr:.3f})')

plt.suptitle('Patient-Level Recalibration', fontsize=13)
plt.tight_layout()
plt.savefig('/kaggle/working/patient_recalibration.png', bbox_inches='tight')
plt.show()

# ── Summary ──────────────────────────────────────────────────────────────
print(f'\n{"═" * 60}')
print(f'PATIENT-LEVEL RECALIBRATION RESULTS')
print(f'{"═" * 60}')
print(f'  Method: Isotonic Regression (5-fold CV)')
print(f'\n  {"Metric":<20s}  {"Before":>10s}  {"After":>10s}  {"Δ":>10s}')
print(f'  {"─"*20}  {"─"*10}  {"─"*10}  {"─"*10}')
print(f'  {"ECE":<20s}  {patient_ece_before:>10.5f}  {patient_ece_after:>10.5f}  {patient_ece_after - patient_ece_before:>+10.5f}')
print(f'  {"AUC":<20s}  {best_patient_auc:>10.5f}  {recal_auc:>10.5f}  {recal_auc - best_patient_auc:>+10.5f}')
print(f'  {"Sensitivity":<20s}  {sens:>10.4f}  {sens_r:>10.4f}  {sens_r - sens:>+10.4f}')
print(f'  {"Specificity":<20s}  {spec:>10.4f}  {spec_r:>10.4f}  {spec_r - spec:>+10.4f}')
print(f'  {"PPV":<20s}  {ppv:>10.4f}  {ppv_r:>10.4f}  {ppv_r - ppv:>+10.4f}')
print(f'  {"F1":<20s}  {f1:>10.4f}  {f1_r:>10.4f}  {f1_r - f1:>+10.4f}')
print(f'  {"Threshold":<20s}  {best_patient_thr:>10.4f}  {recal_thr:>10.4f}  {recal_thr - best_patient_thr:>+10.4f}')
print(f'  {"FP (unnec. escal.)":<20s}  {fp:>10d}  {fp_r:>10d}  {fp_r - fp:>+10d}')
print(f'  {"FN (missed)":<20s}  {fn:>10d}  {fn_r:>10d}  {fn_r - fn:>+10d}')
print(f'{"═" * 60}')

# Save recalibration results
import joblib
joblib.dump(ir_patient, '/kaggle/working/patient_isotonic_regressor.pkl')

recal_results = {
    'method': 'isotonic_5fold_cv',
    'ece_before': round(patient_ece_before, 5),
    'ece_after':  round(patient_ece_after, 5),
    'auc_before': round(best_patient_auc, 5),
    'auc_after':  round(recal_auc, 5),
    'threshold':  round(recal_thr, 4),
    'sensitivity': round(sens_r, 4),
    'specificity': round(spec_r, 4),
    'ppv': round(ppv_r, 4),
    'f1':  round(f1_r, 4),
}
with open('/kaggle/working/patient_recalibration.json', 'w') as f:
    json.dump(recal_results, f, indent=2)

print('\nSaved: patient_isotonic_regressor.pkl, patient_recalibration.json, patient_recalibration.png')

In [ ]:
# ── 12. Training Stability Summary ─────────────────────────────────────────
# Load training metrics log from NB03 to demonstrate convergence.

metrics_log = pd.read_csv(METRICS_LOG_PATH)

print('Training History (all epochs):')
print(metrics_log[['epoch', 'train_loss', 'val_loss', 'val_auc',
                    'sensitivity', 'specificity', 'f1']].to_string(index=False))

# ── Convergence evidence: final epochs ───────────────────────────────────
# Show AUC stability in the plateau region
plateau_start = max(0, len(metrics_log) - 7)  # last 7 epochs
plateau = metrics_log.iloc[plateau_start:]

print(f'\n{"═" * 55}')
print(f'CONVERGENCE ANALYSIS (epochs {plateau.iloc[0]["epoch"]:.0f}–'
      f'{plateau.iloc[-1]["epoch"]:.0f})')
print(f'{"═" * 55}')

print(f'\n  {"Epoch":>5s}  {"AUC":>8s}  {"Val Loss":>9s}  {"ΔAUC":>8s}')
print(f'  {"─"*5}  {"─"*8}  {"─"*9}  {"─"*8}')
prev_auc = None
for _, row in plateau.iterrows():
    delta = f'{row["val_auc"] - prev_auc:+.5f}' if prev_auc is not None else '    —'
    print(f'  {row["epoch"]:5.0f}  {row["val_auc"]:8.5f}  {row["val_loss"]:9.5f}  {delta:>8s}')
    prev_auc = row['val_auc']

auc_std = plateau['val_auc'].std()
auc_range = plateau['val_auc'].max() - plateau['val_auc'].min()
best_epoch = metrics_log.loc[metrics_log['val_auc'].idxmax(), 'epoch']

print(f'\n  AUC std  : {auc_std:.5f}')
print(f'  AUC range: {auc_range:.5f}')
print(f'  Best epoch: {best_epoch:.0f}')
print(f'\n  Conclusion: Model converged between epochs '
      f'{plateau.iloc[0]["epoch"]:.0f}–{plateau.iloc[-1]["epoch"]:.0f}.')
print(f'  AUC oscillates within ±{auc_range/2:.4f}, indicating the '
      f'architecture has saturated.')

# ── Learning curves plot ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(metrics_log['epoch'], metrics_log['train_loss'], 'o-', label='Train')
axes[0].plot(metrics_log['epoch'], metrics_log['val_loss'], 's-', label='Val')
axes[0].set(title='Loss', xlabel='Epoch', ylabel='BCE Loss')
axes[0].legend()

axes[1].plot(metrics_log['epoch'], metrics_log['val_auc'], 'o-', color='tab:orange')
axes[1].axhline(metrics_log['val_auc'].max(), color='red', linestyle='--',
                alpha=0.5, label=f'Best={metrics_log["val_auc"].max():.5f}')
axes[1].fill_between(plateau['epoch'], plateau['val_auc'].min(),
                     plateau['val_auc'].max(), alpha=0.15, color='orange',
                     label='Plateau region')
axes[1].set(title='Validation AUC', xlabel='Epoch', ylabel='AUC')
axes[1].set_ylim([max(0.8, metrics_log['val_auc'].min() - 0.02), 1.0])
axes[1].legend(fontsize=8)

axes[2].plot(metrics_log['epoch'], metrics_log['sensitivity'], 'o-', label='Sensitivity')
axes[2].plot(metrics_log['epoch'], metrics_log['specificity'], 's-', label='Specificity')
axes[2].plot(metrics_log['epoch'], metrics_log['f1'], '^-', label='F1')
axes[2].set(title='Metrics Over Training', xlabel='Epoch')
axes[2].legend(fontsize=8)

plt.suptitle('Training Convergence', fontsize=13)
plt.tight_layout()
plt.savefig('/kaggle/working/training_convergence.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 13. Capacity Ceiling Reflection ────────────────────────────────────────

best_auc = metrics_log['val_auc'].max()
best_ep  = int(metrics_log.loc[metrics_log['val_auc'].idxmax(), 'epoch'])

capacity_text = f"""
{'═' * 65}
ARCHITECTURAL CAPACITY ANALYSIS
{'═' * 65}

Model             : EfficientNet-B0 (5.3M parameters)
Best slice AUC    : {best_auc:.5f} (epoch {best_ep})
Patient-level AUC : {best_patient_auc:.5f} ({best_patient_strategy} aggregation)
Recalibrated AUC  : {recal_auc:.5f} (isotonic 5-fold CV at patient level)
Training regime   : {len(metrics_log)} epochs, early stopped at epoch {int(metrics_log['epoch'].iloc[-1])}

CAPACITY CEILING:
  EfficientNet-B0 saturates around ~{best_auc:.3f} AUC under our current
  preprocessing (3-channel CT windowing) and training scheme (binary "any"
  label, slice-level, weighted sampling, discriminative LR).

  Evidence: AUC oscillated within ±{auc_range/2:.4f} over the final
  {len(plateau)} epochs with no monotonic improvement. Additional epochs
  would not improve performance — the architecture has reached its limit.

CALIBRATION ASSESSMENT:
  Temperature scaling was chosen for deployment despite worsening global
  ECE ({raw_ece:.5f} → {cal_ece:.5f}). Rationale:
    • High-confidence band ECE improved ({hc_ece_raw:.5f} → {hc_ece_cal:.5f})
    • Brier score improved ({raw_brier:.5f} → {cal_brier:.5f})
    • For triage, HC-band reliability matters more than global ECE
  Patient-level recalibration via isotonic regression (5-fold CV) was
  applied to correct max-pooling probability inflation.

KNOWN LIMITATIONS:
  1. Slice-level only: no spatial context across adjacent CT slices
  2. Binary "any" label: does not model 5 hemorrhage subtypes
  3. Single split: AUC estimate has variance ~±0.005 without cross-validation
  4. Global ECE worsened after temp scaling (HC-band improved — acceptable)
  5. {fp} false positive patient escalations at current threshold

PATHS TO IMPROVEMENT (future work, separate from this project):
  ┌──────────────────────────────────┬────────────┬──────────────────┐
  │ Approach                         │ Effort     │ Expected Gain    │
  ├──────────────────────────────────┼────────────┼──────────────────┤
  │ Stronger backbone (B3/B4/B7)    │ Medium     │ +0.5–1.0% AUC   │
  │ Advanced augmentation (MixUp)   │ Low        │ +0.2–0.5%        │
  │ Multi-label (5 subtypes + any)  │ High       │ +1–2%            │
  │ Patient-level aggregation+train │ Medium     │ +0.5–1.0%        │
  │ Ensemble (2–3 architectures)    │ Medium     │ +0.3–0.8%        │
  │ Cross-validation (5-fold)       │ High       │ Variance estimate│
  └──────────────────────────────────┴────────────┴──────────────────┘

  These improvements are planned as an enhancement project, separate from
  the current major project scope.
{'═' * 65}
"""
print(capacity_text)

In [ ]:
# ── 14. Deployment Decision ────────────────────────────────────────────────
# One clean declaration of the final selected configuration.

deployment_config = {
    'model': {
        'architecture'     : ARCH,
        'parameters'       : '5.3M',
        'best_epoch'       : best_ep,
        'checkpoint'       : 'best_model.pth',
    },
    'slice_calibration': {
        'method'           : CALIB_METHOD,
        'temperature'      : TEMPERATURE,
        'global_ece'       : {'raw': round(raw_ece, 5), 'calibrated': round(cal_ece, 5),
                              'effect': 'Worsened (+0.026) — acceptable tradeoff'},
        'hc_band_ece'      : {'raw': round(hc_ece_raw, 5), 'calibrated': round(hc_ece_cal, 5),
                              'effect': 'Improved (critical for triage)'},
        'brier_score'      : {'raw': round(raw_brier, 5), 'calibrated': round(cal_brier, 5),
                              'effect': 'Improved'},
        'note'             : ('Temperature scaling worsened global ECE slightly but improved '
                              'high-confidence band ECE and Brier score. For triage deployment, '
                              'high-confidence calibration is the critical metric.'),
    },
    'patient_calibration': {
        'method'           : 'Isotonic regression (5-fold CV)',
        'ece_before'       : round(patient_ece_before, 5),
        'ece_after'        : round(patient_ece_after, 5),
        'threshold'        : round(recal_thr, 4),
    },
    'thresholds': {
        'slice_threshold'  : BASE_THRESHOLD,
        'patient_threshold': round(recal_thr, 4),
        'high_conf_band'   : HIGH_THRESHOLD,
        'low_conf_band'    : LOW_THRESHOLD,
    },
    'aggregation': {
        'strategy'         : best_patient_strategy,
        'patient_auc'      : round(best_patient_auc, 5),
    },
    'performance': {
        'slice_auc'          : round(slice_auc, 5),
        'patient_auc'        : round(best_patient_auc, 5),
        'patient_auc_recal'  : round(recal_auc, 5),
        'patient_sensitivity': round(sens_r, 4),
        'patient_specificity': round(spec_r, 4),
        'patient_ppv'        : round(ppv_r, 4),
        'patient_f1'         : round(f1_r, 4),
    },
    'caveats': [
        'Single train/val split — no cross-validation variance estimate',
        'Temperature scaling worsened global ECE but improved HC-band ECE and Brier',
        'Binary "any" label — no subtype classification',
        'Not cleared for standalone diagnostic use',
    ],
}

# Pretty-print deployment card
print(f"""
{'█' * 65}
  FINAL DEPLOYMENT CONFIGURATION
{'█' * 65}

  Model
    Architecture       : {ARCH}
    Best epoch         : {best_ep}
    Checkpoint         : best_model.pth

  Slice-Level Calibration
    Method             : {CALIB_METHOD.title()} Scaling (T={TEMPERATURE:.4f})
    Global ECE         : {raw_ece:.5f} → {cal_ece:.5f} (worsened; acceptable)
    HC-band ECE        : {hc_ece_raw:.5f} → {hc_ece_cal:.5f} (improved; critical)
    Brier              : {raw_brier:.5f} → {cal_brier:.5f} (improved)

  Patient-Level Recalibration
    Method             : Isotonic Regression (5-fold CV)
    ECE                : {patient_ece_before:.5f} → {patient_ece_after:.5f}

  Thresholds
    Slice threshold    : {BASE_THRESHOLD:.4f}
    Patient threshold  : {recal_thr:.4f} (recalibrated)
    High-conf band     : ≥ {HIGH_THRESHOLD:.2f}
    Low-conf band      : < {LOW_THRESHOLD:.2f}

  Patient-Level Aggregation
    Strategy           : {best_patient_strategy}

  Performance Summary
    ┌────────────────────┬─────────────┬──────────────┬──────────────┐
    │ Metric             │ Slice-Level │ Patient (raw)│ Patient (cal)│
    ├────────────────────┼─────────────┼──────────────┼──────────────┤
    │ AUC                │ {slice_auc:>11.5f} │ {best_patient_auc:>12.5f} │ {recal_auc:>12.5f} │
    │ Sensitivity        │      —      │ {sens:>12.4f} │ {sens_r:>12.4f} │
    │ Specificity        │      —      │ {spec:>12.4f} │ {spec_r:>12.4f} │
    │ PPV                │      —      │ {ppv:>12.4f} │ {ppv_r:>12.4f} │
    │ F1                 │      —      │ {f1:>12.4f} │ {f1_r:>12.4f} │
    │ FP (unnec. escal.) │      —      │ {fp:>12d} │ {fp_r:>12d} │
    │ FN (missed)        │      —      │ {fn:>12d} │ {fn_r:>12d} │
    └────────────────────┴─────────────┴──────────────┴──────────────┘

  Caveats
    • Single split — AUC variance ~±0.005 without CV
    • Temp scaling worsened global ECE; improved HC-band ECE + Brier
    • Binary "any" label — 5 subtypes not modeled
    • Screening aid only — not for standalone diagnosis

{'█' * 65}
""")

# Save deployment config
with open('/kaggle/working/deployment_config.json', 'w') as f:
    json.dump(deployment_config, f, indent=2)

print('Saved: deployment_config.json')

In [ ]:
# ── 15. Cleanup + final output summary ───────────────────────────────────
grad_cam.remove()

import glob
report_files = glob.glob(str(REPORTS_DIR / '*.json'))
print(f'\nTotal JSON reports saved: {len(report_files)}')
print(f'Grad-CAM overlays saved : {len(glob.glob(str(REPORTS_DIR / "*_gradcam.png")))}')
print(f'Summary CSV             : /kaggle/working/report_summary.csv')
print(f'Patient-level results   : /kaggle/working/patient_level_results.json')
print(f'Patient recalibration   : /kaggle/working/patient_recalibration.json')
print(f'Patient isotonic model  : /kaggle/working/patient_isotonic_regressor.pkl')
print(f'Deployment config       : /kaggle/working/deployment_config.json')
print(f'Training convergence    : /kaggle/working/training_convergence.png')
print(f'Patient-level analysis  : /kaggle/working/patient_level_analysis.png')
print(f'Patient recalibration   : /kaggle/working/patient_recalibration.png')
print()
print('All notebooks complete. Full pipeline:')
print('  01_eda.ipynb              → Dataset exploration + windowing demo')
print('  02_preprocess_cache.ipynb → DICOM → NPY cache (commit output)')
print('  03_train_session.ipynb    → Training with session chaining')
print('  04_ablations.ipynb        → Architecture + preprocessing + augmentation ablations')
print('  05_gradcam.ipynb          → Grad-CAM heatmaps + occlusion sanity check')
print('  06_calibration.ipynb      → Temperature scaling + 3-band triage system')
print('  07_report.ipynb           → Clinical reports + patient-level analysis (this notebook)')

In [ ]:
# ── HEALTH CHECK — automated output validation ────────────────────────────

errors = []

# Check reports directory
import glob as _glob
json_reports = _glob.glob(str(REPORTS_DIR / '*.json'))
cam_files    = _glob.glob(str(REPORTS_DIR / '*_gradcam.png'))

if len(json_reports) < N_REPORTS:
    errors.append(f'Only {len(json_reports)} JSON reports — expected {N_REPORTS}')

if len(cam_files) < N_REPORTS:
    errors.append(f'Only {len(cam_files)} Grad-CAM overlays — expected {N_REPORTS}')

# Check summary CSV
if not os.path.exists('/kaggle/working/report_summary.csv'):
    errors.append('report_summary.csv is MISSING')

# Check patient-level results
if not os.path.exists('/kaggle/working/patient_level_results.json'):
    errors.append('patient_level_results.json is MISSING')
else:
    with open('/kaggle/working/patient_level_results.json') as f:
        pl = json.load(f)
    if pl.get('patient_auc', 0) < 0.5:
        errors.append(f'Patient-level AUC={pl["patient_auc"]:.4f} is suspiciously low')

# Check patient-level recalibration outputs
for recal_file in ['patient_recalibration.json', 'patient_isotonic_regressor.pkl']:
    if not os.path.exists(f'/kaggle/working/{recal_file}'):
        errors.append(f'{recal_file} is MISSING')

if os.path.exists('/kaggle/working/patient_recalibration.json'):
    with open('/kaggle/working/patient_recalibration.json') as f:
        recal = json.load(f)
    if recal.get('ece_after', 1.0) > recal.get('ece_before', 0.0) + 0.05:
        errors.append(f'Patient recalibration ECE worsened significantly: '
                      f'{recal["ece_before"]:.4f} → {recal["ece_after"]:.4f}')

# Check deployment config
if not os.path.exists('/kaggle/working/deployment_config.json'):
    errors.append('deployment_config.json is MISSING')

# Check new plots
for plot in ['patient_level_analysis.png', 'training_convergence.png',
             'patient_recalibration.png']:
    if not os.path.exists(f'/kaggle/working/{plot}'):
        errors.append(f'Missing plot: {plot}')

# Validate report structure (sample one)
if json_reports:
    with open(json_reports[0]) as f:
        sample_rep = json.load(f)
    required_keys = ['report_id', 'image_id', 'prediction', 'triage', 'disclaimer']
    for k in required_keys:
        if k not in sample_rep:
            errors.append(f'Report missing key: {k}')
    if 'disclaimer' in sample_rep and len(sample_rep['disclaimer']) < 50:
        errors.append('Disclaimer text is too short — possible truncation')

health = {
    'notebook'                 : '07_report',
    'status'                   : 'PASS' if not errors else 'FAIL',
    'errors'                   : errors,
    'n_reports'                : len(json_reports),
    'n_gradcam'                : len(cam_files),
    'summary_csv'              : os.path.exists('/kaggle/working/report_summary.csv'),
    'patient_level_auc'        : round(best_patient_auc, 5),
    'patient_aggregation'      : best_patient_strategy,
    'slice_auc'                : round(slice_auc, 5),
    'patient_recalibration'    : os.path.exists('/kaggle/working/patient_recalibration.json'),
    'deployment_config_saved'  : os.path.exists('/kaggle/working/deployment_config.json'),
}

with open('/kaggle/working/health_check_nb07.json', 'w') as f:
    json.dump(health, f, indent=2)

if errors:
    print('❌ HEALTH CHECK FAILED:')
    for e in errors:
        print(f'   • {e}')
else:
    print('✅ HEALTH CHECK PASSED')
    print(f'   Reports              : {len(json_reports)} JSON + {len(cam_files)} Grad-CAM')
    print(f'   Summary CSV          : saved')
    print(f'   Slice-level AUC      : {slice_auc:.5f}')
    print(f'   Patient-level AUC    : {best_patient_auc:.5f} ({best_patient_strategy})')
    print(f'   Patient recalibration: saved (isotonic 5-fold CV)')
    print(f'   Deployment config    : saved')
    print(f'   Pipeline             : COMPLETE ✓')
    print()
    print('All 7 notebooks have completed successfully.')
    print('Check health_check_nb*.json files for per-notebook validation results.')